In [ ]:
!pip install mne

In [ ]:
from scipy.io import loadmat
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import mne
from mne.preprocessing import ICA
import pandas as pd

In [ ]:
def procesamiento(raw):
  ''' Utiliza las fecuencias de corte:
  inferion a 1Hz - > elimina deriva de línea base, DC offset y artefactos
   de movimiento de baja frecuencia'

   Superior a 40Hz-> elimina ruido de línea eléctrica y
   artefactos mioeléctricos de alta frecuencia'

   '''

  raw.filter(l_freq=1.0, h_freq=40.0, method='fir',
       	fir_window='hamming', phase='zero')

  '''Resta en cada instante la media de los canales ->mejora relacion'''
  raw.set_eeg_reference(ref_channels='average', projection=False)

  return raw

def ica(signal_filtered):

      ''' Divide la señal en 20 componentes(combinaciones lienales de canales) y los compara con una señal ocular
      la cual toma directamente de electrodos en esta zona o cercanas como los Fp1 y Fp2

      input: señal filtrada (preprocesamiento) Array
      output: señal limpia de ruido ocular  Array'''

      ica = ICA(n_components=20, random_state=42, method='fastica')
      ica.fit(signal_filtered, picks='eeg')

      # Identificar componentes de artefacto ocular (comparacion con criterio estadistico de 3.0)
      eog_idx, scores = ica.find_bads_eog(signal_filtered, threshold=3.0)

      ica.exclude = eog_idx
    # Toma los componentes ica y recostruye la señal excluyendo los que se encuentran en ica.exclude
      raw_clean = ica.apply(raw.copy())

      return raw_clean

def Graficar(raw,signal_filtered,raw_clean):
  picks_c3 = ['C3']
  start = 10

  raw.plot(picks=picks_c3, start=start, duration=5, title='Bruta')
  signal_filtered.plot(picks=picks_c3, start=start, duration=5, title='Filtrada')
  raw_clean.plot(picks=picks_c3, start=start, duration=5, title='ICA')



def cargar_datos(path,n):
    '''Permite cargar los datos de n sujetos de 6 runs segun lo solicitado
    y almacenar la data en epocas y en grupos segun tareas y extremidades
     input = path STR
              n INT
     output = datos DICT,canales LIST
     '''

    datos = {}

    runs_validos = [3,4,7,8,11,12]
    signal=[]


    for j in range(1, n+1):

        sujeto = f"S{j:03d}"#minimo de 3 digitos -> completa con ceros a la izquierda
        datos[sujeto] = {
            "real": {"T1": [], "T2": []}, #listas para cada extremidad
            "imaginado": {"T1": [], "T2": []}
        }

        for i in runs_validos:

            run = f"R{i:02d}"
            archivo = f"{path}/{sujeto}{run}.edf"
            raw = mne.io.read_raw_edf(archivo, preload=True)
            canales = raw.ch_names
            events, event_id = mne.events_from_annotations(raw)
            signal.append(raw)

      return signal,canales,events,event_id

    def segmentacionEpocas(signal,canales,events,event_id):
      for i in signal:
         epochs = mne.Epochs(
                i,
                events,
                event_id={'T1': event_id['T1'], 'T2': event_id['T2']},
                tmin=-1.0, #rangos del marcador
                tmax=4,
                baseline=None,
                preload=True
            )
        return epochs
